# 03 — Model Training & Evaluation
Train and compare five classifiers (Random Forest, Logistic Regression, Neural Network, XGBoost baseline, XGBoost tuned) on 12 audio features across 8 genres (2,000 samples each).
The feature importance output from Random Forest tells us what to cut in a future iteration.

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Cell 2 — Load processed arrays and label encoder
X_train = np.load("../data/processed/X_train.npy")
X_test  = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")
le      = joblib.load("../models/label_encoder.pkl")

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {len(le.classes_)} genres")

In [ ]:
# Cell 3 — Train RandomForestClassifier
# n_estimators=200: 200 trees — good balance of accuracy vs training time
# n_jobs=-1: use all CPU cores to parallelize training
# random_state=42: reproducible results
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# Cell 4 — Evaluate: classification report (F1, precision, recall per genre)
# F1 is our primary metric because genres are imbalanced in general;
# here they're balanced, but F1 still gives a clearer per-class picture than accuracy
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Cell — Logistic Regression baseline
# saga solver handles multinomial (8-class) classification efficiently at this scale.
# max_iter=2000 because convergence on multi-class problems can take more iterations than the default 100.
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=42,
    n_jobs=-1
)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
lr_macro  = f1_score(y_test, y_pred_lr, average='macro')

print("=== Logistic Regression — Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))
print(f"Logistic Regression macro F1 : {lr_macro:.4f}")

In [ ]:
# Cell — Neural Network (MLP) baseline
# MLPClassifier is sklearn's multi-layer perceptron — a basic feedforward NN.
# Two hidden layers (256 → 128) gives enough capacity for 8-class classification
# without being so deep it needs a GPU or hours to train.
# early_stopping=True holds out 10% of training data as a validation set and
# stops training when val score stops improving — prevents overfitting.
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=500,
    early_stopping=True,
    random_state=42,
    verbose=False
)
nn.fit(X_train, y_train)

y_pred_nn = nn.predict(X_test)
nn_macro  = f1_score(y_test, y_pred_nn, average='macro')

print("=== Neural Network (MLP) — Classification Report ===")
print(classification_report(y_test, y_pred_nn, target_names=le.classes_))
print(f"Neural Network macro F1 : {nn_macro:.4f}")

In [ ]:
# Cell — Baseline XGBoost (default parameters)
# We train once with defaults before tuning so we can see how much the
# hyperparameter search actually helps. eval_metric silences a warning
# that XGBoost would otherwise print about log-loss during training.
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

xgb_baseline = XGBClassifier(
    eval_metric='mlogloss',   # avoids deprecation warning in newer XGBoost
    random_state=42,
    n_jobs=-1
)
xgb_baseline.fit(X_train, y_train)

y_pred_xgb_baseline = xgb_baseline.predict(X_test)
xgb_baseline_f1 = f1_score(y_test, y_pred_xgb_baseline, average='macro')
print(f"Baseline XGBoost macro F1 : {xgb_baseline_f1:.4f}")

In [ ]:
# Cell — RandomizedSearchCV: find best XGBoost hyperparameters
# Instead of manually guessing good values, we let the search try 30 random
# combinations and pick whichever scores highest on macro F1.
# n_iter=30: 30 random combinations — good coverage without running all day
# cv=3: 3-fold cross-validation — each combo is evaluated 3 times for reliability
# n_jobs=-1 on the search (not the estimator) avoids nested parallelism on macOS
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [3, 5, 7, 9],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=1),
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,    # parallelize the search itself across CPU cores
    verbose=1     # prints progress so you know it's running
)
xgb_search.fit(X_train, y_train)

print("\nBest parameters found:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param:<20} {value}")
print(f"\nBest cross-val macro F1 : {xgb_search.best_score_:.4f}")

In [ ]:
# Cell — Evaluate tuned XGBoost on the test set
# xgb_search.best_estimator_ is the model that scored highest in the search.
# We run it on X_test (data it has never seen) for an honest evaluation.
from sklearn.metrics import classification_report

best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print("=== Tuned XGBoost — Classification Report ===")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

In [ ]:
# Cell — Head-to-head bar chart: all 5 models
# classification_report with output_dict=True lets us pull per-genre F1 for each model.
import pandas as pd

report_rf       = classification_report(y_test, y_pred,              target_names=le.classes_, output_dict=True)
report_lr       = classification_report(y_test, y_pred_lr,           target_names=le.classes_, output_dict=True)
report_nn       = classification_report(y_test, y_pred_nn,           target_names=le.classes_, output_dict=True)
report_xgb_base = classification_report(y_test, y_pred_xgb_baseline, target_names=le.classes_, output_dict=True)
report_xgb      = classification_report(y_test, y_pred_xgb,          target_names=le.classes_, output_dict=True)

genres = list(le.classes_)
rf_f1s       = [report_rf[g]['f1-score']       for g in genres]
lr_f1s       = [report_lr[g]['f1-score']       for g in genres]
nn_f1s       = [report_nn[g]['f1-score']       for g in genres]
xgb_base_f1s = [report_xgb_base[g]['f1-score'] for g in genres]
xgb_f1s      = [report_xgb[g]['f1-score']      for g in genres]

rf_macro       = f1_score(y_test, y_pred,              average='macro')
xgb_base_macro = f1_score(y_test, y_pred_xgb_baseline, average='macro')
xgb_macro      = f1_score(y_test, y_pred_xgb,          average='macro')

x     = np.arange(len(genres))
width = 0.15

fig, ax = plt.subplots(figsize=(22, 6))
ax.bar(x - 2*width, rf_f1s,       width, label=f'Random Forest       (macro F1 = {rf_macro:.3f})',       color='steelblue')
ax.bar(x - 1*width, lr_f1s,       width, label=f'Logistic Regression  (macro F1 = {lr_macro:.3f})',      color='mediumseagreen')
ax.bar(x,           nn_f1s,       width, label=f'Neural Network       (macro F1 = {nn_macro:.3f})',      color='mediumpurple')
ax.bar(x + 1*width, xgb_base_f1s, width, label=f'XGBoost baseline     (macro F1 = {xgb_base_macro:.3f})', color='sandybrown')
ax.bar(x + 2*width, xgb_f1s,      width, label=f'XGBoost tuned        (macro F1 = {xgb_macro:.3f})',     color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels(genres, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.0)
ax.set_title('Model Comparison — F1 per Genre (5 Models)', fontsize=14)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nRandom Forest       macro F1 : {rf_macro:.4f}")
print(f"Logistic Regression macro F1 : {lr_macro:.4f}")
print(f"Neural Network      macro F1 : {nn_macro:.4f}")
print(f"XGBoost baseline    macro F1 : {xgb_base_macro:.4f}")
print(f"XGBoost tuned       macro F1 : {xgb_macro:.4f}")

In [ ]:
# Cell — Save the winning model
# Pick the highest macro F1 across all five models trained so far.
# best_model.pkl is what notebook 04 loads for real-world predictions.
scores = {
    'Random Forest'      : (rf,           rf_macro),
    'Logistic Regression': (lr,           lr_macro),
    'Neural Network'     : (nn,           nn_macro),
    'XGBoost baseline'   : (xgb_baseline, xgb_base_macro),
    'XGBoost tuned'      : (best_xgb,     xgb_macro),
}

winner_name, (winner, winner_score) = max(scores.items(), key=lambda x: x[1][1])

joblib.dump(winner, '../models/best_model.pkl')
print(f"Winner   : {winner_name}")
print(f"Macro F1 : {winner_score:.4f}")
print(f"Saved to : models/best_model.pkl")

print("\nAll model scores:")
for name, (_, score) in sorted(scores.items(), key=lambda x: -x[1][1]):
    marker = " <-- winner" if name == winner_name else ""
    print(f"  {name:<25} {score:.4f}{marker}")

In [ ]:
# Cell 5 — Confusion matrix heatmap
# 8 genres is small enough to show per-cell counts with annot=True.
# Diagonal = correct predictions; off-diagonal = misclassifications.
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title("Confusion Matrix — Random Forest (8 Genres)", fontsize=14)
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Feature importance bar chart
# Which of the 12 features does the Random Forest rely on most?
# Low-importance features are candidates to cut in the next iteration.
ALL_FEATURES = [
    'popularity', 'duration_ms', 'mode',
    'danceability', 'energy', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'loudness',
    'speechiness', 'acousticness'
]

importances = pd.Series(rf.feature_importances_, index=ALL_FEATURES).sort_values()

plt.figure(figsize=(9, 6))
importances.plot(kind="barh", color="steelblue")
plt.title("Feature Importances — Random Forest (12 Features)", fontsize=14)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature importances (descending):")
for feat, imp in importances.sort_values(ascending=False).items():
    print(f"  {feat:<20} {imp:.4f}")

In [ ]:
# Cell 7 — Save trained model
joblib.dump(rf, "../models/random_forest.pkl")
print("Model saved to models/random_forest.pkl")